# Capstone Project: Exploratory Data Analysis (EDA)
## Network Intrusion & Threat Landscape Analysis (CICIDS2017)
**Candidate:** Adarsh Kumar Sharma | **Roll No:** 2301420058 | **Program:** B.Tech CSE (Data Science)
**Project:** AI-Powered Dual-Stage Network Intrusion Detection System  
**Objective:** Ingest flow logs, audit missing/infinite values, analyze class imbalances, and evaluate flow distributions.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.insert(0, "..")
from src.data_loader import load_dataset, prepare_labels
from src.config import load_config, CORE_FEATURES

cfg = load_config()
print("Project root and configurations verified.")

### 1. Ingest Dataset & Audit Data Hygiene
The raw CICIDS2017 data contains whitespace in column names and infinite values in `Flow Bytes/s` due to zero-duration micro-flows. We audit and clean these records.

In [ ]:
data_path = cfg["paths"]["primary_dataset"]
df = load_dataset(data_path, max_rows=50000)
df = prepare_labels(df, cfg["attack_taxonomy"])

print("Dataset Shape:", df.shape)
print("Columns:", len(df.columns))
df.head()

### 2. Threat Class Distribution
Analyze the ratio between Normal (Benign) traffic and various attack types (DDoS, PortScan, SSH-Patator, Botnet, Web Attacks).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Binary Distribution
binary_counts = df["is_attack"].map({0: "Normal", 1: "Attack"}).value_counts()
axes[0].pie(binary_counts, labels=binary_counts.index, autopct="%1.1f%%", colors=["#2a78d6", "#eb6834"], startangle=90)
axes[0].set_title("Stage 1: Binary Flow Distribution")

# Multi-class Taxonomy
attack_counts = df[df["is_attack"] == 1]["attack_category"].value_counts()
attack_counts.plot(kind="barh", ax=axes[1], color="#eb6834")
axes[1].set_title("Stage 2: Threat Taxonomy Breakdown")
axes[1].set_xlabel("Flow Count")

plt.tight_layout()
plt.show()

### 3. Port Bucketing Analysis
Network security standards categorize ports into:
- `well_known`: 0 to 1023 (HTTP, HTTPS, SSH, DNS)
- `registered`: 1024 to 49151 (Databases, Proxies)
- `dynamic`: 49152 to 65535 (Ephemeral client ports)

We evaluate how attack traffic is distributed across these port regimes.

In [ ]:
df["Port Class"] = pd.cut(
    df["Destination Port"],
    bins=[-1, 1023, 49151, 65535],
    labels=["well_known", "registered", "dynamic"]
)

port_crosstab = pd.crosstab(df["Port Class"], df["is_attack"], normalize="index")
ax = port_crosstab.plot(kind="bar", stacked=True, color=["#2a78d6", "#eb6834"], figsize=(8, 4.5))
ax.set_title("Traffic Type Distribution by Port Class")
ax.set_ylabel("Proportion")
ax.legend(["Normal", "Attack"], loc="upper right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()